# 1차시: 싱글 에이전트

LangChain 기반 AI 에이전트 구축 실습 노트북입니다.

## 목차
1. LangChain 기본 사용법
2. Planning & CoT (계획과 추론)
3. Tool Use (도구 사용)
4. 메모리
5. RAG (검색 증강 생성)
6. Reflexion (자기 평가 및 개선)
7. LangGraph 통합

---
## 1. LangChain 기본 사용법

LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다.

`OPENAI_API_KEY` 환경 변수가 설정되어 있으면 자동으로 사용됩니다.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-..."

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0.7)

response = llm.invoke("안녕하세요!")
print(response.content)

---
## 2. Planning & CoT (계획과 추론)

### Planning이란?
복잡한 목표를 **작은 하위 작업으로 분해**하는 능력입니다.
- 큰 작업을 실행 가능한 단위로 분해 (Task Decomposition)
- 중간 목표 설정으로 진행 상황 추적 (Subgoal Setting)
- 작업 간 의존성 고려하여 실행 순서 결정 (Ordering)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

planning_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 작업 계획을 수립하는 AI입니다.
    사용자의 요청을 분석하여 단계별 실행 계획을 세우세요.

    출력 형식:
    1. [단계 1 설명]
    2. [단계 2 설명]
    ..."""),
    ("human", "{task}")
])

planning_chain = planning_prompt | llm
plan = planning_chain.invoke({"task": "최신 AI 논문을 찾아 요약해줘"})
print(plan.content)

### Chain of Thought (CoT)

LLM이 **단계별로 추론**하도록 유도하는 프롬프팅 기법입니다.

| 직접 답변 | Chain of Thought |
|:---:|:---|
| 질문 → 답변 | 질문 → 추론1 → 추론2 → ... → 답변 |
| 오류 가능성 높음 | 추론 과정 검증 가능 |

CoT는 **한 번의 LLM 호출**로 추론 과정과 최종 답변이 함께 나옵니다.

In [ ]:
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 논리적으로 사고하는 AI입니다.
    문제를 해결할 때 다음 단계를 따르세요:

    1. 문제 이해: 주어진 정보 정리
    2. 계획 수립: 해결 방법 구상
    3. 단계별 실행: 각 단계를 명시적으로 수행
    4. 검증: 답이 맞는지 확인
    5. 최종 답변: 결론 제시"""),
    ("human", "{question}")
])

cot_chain = cot_prompt | llm
response = cot_chain.invoke({"question": "가게에 사과 23개가 있다. 7개를 팔고 15개를 더 받았다면?"})
print(response.content)

---
## 3. Tool Use (도구 사용)

에이전트가 **외부 도구/API**를 호출하여 기능을 확장하는 능력입니다.

- LLM만으로는 불가능한 작업 수행 (검색, 계산, 파일 처리 등)
- 실시간 정보 접근 및 외부 시스템 연동
- 에이전트 능력 확장의 기반

### 툴 정의 방법

`@tool` 데코레이터를 사용하여 함수를 도구로 정의합니다. docstring이 도구 설명으로 사용됩니다.

In [ ]:
from langchain_core.tools import tool

@tool
def search_weather(city: str) -> str:
    """도시의 현재 날씨를 검색합니다."""
    # 실제로는 날씨 API 호출
    return f"{city}의 날씨: 맑음, 20°C"

@tool
def calculate(expression: str) -> str:
    """수학 표현식을 계산합니다."""
    try:
        result = eval(expression)
        return f"결과: {result}"
    except Exception:
        return "계산 오류"

tools = [search_weather, calculate]

print(f"정의된 도구: {[t.name for t in tools]}")

### 에이전트에 툴 연결

`create_agent`를 사용하면 ReAct 패턴의 에이전트를 쉽게 생성할 수 있습니다.

에이전트는 사용자 요청을 분석하여 **자동으로 적절한 도구를 선택**하고 호출합니다.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(llm, tools)

result = agent.invoke({
    "messages": [("user", "서울 날씨 알려주고, 234 * 567 계산해줘")]
})

print(result["messages"][-1].content)

---
## 4. 메모리

LLM은 기본적으로 **상태가 없음** (Stateless)입니다.
- 매 요청마다 이전 대화 내용을 기억하지 못함
- 사용자 선호도나 과거 작업 결과 활용 불가

메모리를 통해 **맥락 유지**, **개인화**, **학습**을 구현할 수 있습니다.

### 메모리 구현

`MemorySaver`를 사용하면 대화 컨텍스트를 유지할 수 있습니다. `thread_id`로 세션을 구분합니다.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent = create_agent(llm, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "user-123"}}

result = agent.invoke({"messages": [("user", "내 이름은 철수야")]}, config)
print(result["messages"][-1].content)

result = agent.invoke({"messages": [("user", "내 이름이 뭐라고 했지?")]}, config)
print(result["messages"][-1].content)

---
## 5. RAG (검색 증강 생성)

**검색(Retrieval)** + **생성(Generation)** 결합

- 외부 문서에서 관련 정보를 검색하여 LLM에 제공
- LLM이 검색된 컨텍스트를 기반으로 답변 생성
- 학습 데이터에 없는 최신/도메인 지식 활용 가능

### RAG 구현 - 문서 인덱싱

1. 문서를 로드하고
2. 적절한 크기로 분할(chunking)한 후
3. 임베딩하여 벡터 저장소에 저장합니다.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. 문서 로드
loader = TextLoader("knowledge_base.txt")
documents = loader.load()

# 2. 청크 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
)
chunks = splitter.split_documents(documents)

# 3. 벡터 저장소 생성
vectorstore = Chroma.from_documents(chunks, OpenAIEmbeddings())

print(f"문서 수: {len(documents)}, 청크 수: {len(chunks)}")

### RAG 구현 - 검색 및 생성

검색기(retriever)를 설정하고, 프롬프트와 LLM을 연결하여 RAG 체인을 구성합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# 검색기 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG 프롬프트
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """다음 컨텍스트를 참고하여 질문에 답하세요.
    컨텍스트에 없는 내용은 "모르겠습니다"라고 답하세요.

    컨텍스트: {context}"""),
    ("human", "{question}")
])

# RAG 체인
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt | llm
)

# RAG 체인 실행
response = rag_chain.invoke("문서의 주요 내용이 뭐야?")
print(response.content)

---
## 6. Reflexion (자기 평가 및 개선)

에이전트가 **자신의 출력을 평가**하고 **반복적으로 개선**하는 능력입니다.

**Reflexion 프로세스:**
1. 초기 응답 생성
2. 응답 평가 (점수 + 피드백)
3. 품질 기준 미달 시 → 피드백 반영하여 재생성
4. 기준 충족 또는 최대 반복 도달 시 → 최종 응답 반환

### 프롬프트 정의

생성, 평가, 개선을 위한 3개의 프롬프트를 정의합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 초기 응답 생성 프롬프트
generator_prompt = ChatPromptTemplate.from_messages([
    ("system", "질문에 상세하고 정확하게 답변하세요."),
    ("human", "{question}")
])

# 평가 프롬프트
evaluator_prompt = ChatPromptTemplate.from_messages([
    ("system", """응답을 평가하고 JSON 형식으로 출력하세요.
    {{"score": 1-10, "feedback": "개선점"}}"""),
    ("human", "질문: {question}\n응답: {response}")
])

# 개선 프롬프트
improver_prompt = ChatPromptTemplate.from_messages([
    ("system", "피드백을 반영하여 응답을 개선하세요."),
    ("human", "질문: {question}\n이전 응답: {response}\n피드백: {feedback}")
])

### Reflexion 실행

응답을 생성하고, 평가하고, 개선하는 반복 루프를 실행합니다.

In [ ]:
import json

def reflexion(question, max_iterations=3):
    # 초기 응답 생성
    response = (generator_prompt | llm | StrOutputParser()).invoke(
        {"question": question}
    )
    print(f"[초기 응답]\n{response}\n")

    for i in range(max_iterations):
        # 평가
        eval_result = (evaluator_prompt | llm | StrOutputParser()).invoke(
            {"question": question, "response": response}
        )
        evaluation = json.loads(eval_result)
        print(f"[{i+1}차 평가] 점수: {evaluation['score']}, 피드백: {evaluation['feedback']}")

        if evaluation["score"] >= 8:
            print("품질 기준 충족!")
            break

        # 개선
        response = (improver_prompt | llm | StrOutputParser()).invoke(
            {"question": question, "response": response,
             "feedback": evaluation["feedback"]}
        )
        print(f"[{i+1}차 개선]\n{response}\n")

    return response

# Reflexion 실행
result = reflexion("파이썬의 장점 3가지를 설명해줘")
print(f"\n[최종 응답]\n{result}")

---
## 7. LangGraph 통합

지금까지 배운 모든 구성요소를 통합합니다:
- **Tool Use**: RAG 검색, 날씨, 계산 도구
- **Memory**: 대화 컨텍스트 유지
- **CoT**: 단계별 추론
- **Reflexion**: 응답 품질 개선

### 도구 정의

In [ ]:
from langchain_core.tools import tool

@tool
def search_knowledge(query: str) -> str:
    """knowledge_base에서 관련 정보를 검색합니다."""
    docs = retriever.invoke(query)
    return "\n".join([d.page_content for d in docs])

@tool
def search_weather(city: str) -> str:
    """도시의 현재 날씨를 검색합니다."""
    return f"{city}의 날씨: 맑음, 20°C"

@tool
def calculate(expression: str) -> str:
    """수학 표현식을 계산합니다."""
    return f"결과: {eval(expression)}"

tools = [search_knowledge, search_weather, calculate]

### CoT 프롬프트 정의

평가와 개선에 사용할 CoT(단계별 추론) 프롬프트를 정의합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# CoT 평가 프롬프트
cot_eval = ChatPromptTemplate.from_messages([
    ("system", """이 응답은 도구(검색/계산)를 사용한 에이전트가 생성했습니다.
    응답에 포함된 수치와 정보는 실제 도구 실행 결과입니다.

    단계별로 분석하세요:
    1. 질문의 요구사항을 모두 충족했는가?
    2. 응답이 간결하고 명확한가?
    3. 불필요한 설명이 있는가?

    주의: 도구가 반환한 데이터의 정확성은 의심하지 마세요.
    JSON 형식으로 출력: {{"score": 1-10, "feedback": "개선점"}}"""),
    ("human", "질문: {question}\n응답: {response}")
])

# CoT 개선 프롬프트
cot_improve = ChatPromptTemplate.from_messages([
    ("system", """단계별로 생각한 후, 최종 개선된 응답만 출력하세요:
    1. 피드백 분석: 어떤 점을 개선해야 하는가?
    2. 개선 방향: 간결성, 명확성 위주로 수정
    3. 응답 다듬기: 핵심 정보(수치, 계산 결과)는 반드시 유지

    주의사항:
    - 원본 응답의 수치와 결과를 절대 변경하지 마세요
    - "검색할 수 없다", "확인이 필요하다" 등의 표현을 추가하지 마세요
    - 분석 과정(1,2,3번)은 출력하지 말고, 최종 응답만 출력하세요"""),
    ("human", "질문: {question}\n응답: {response}\n피드백: {feedback}")
])

### 에이전트 + Reflexion

Memory가 있는 에이전트를 생성하고, CoT 프롬프트를 사용한 Reflexion으로 응답 품질을 개선합니다.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import MemorySaver
import json

memory = MemorySaver()
agent = create_agent(llm, tools, checkpointer=memory)

def agent_with_reflexion(query, config, max_iter=2):
    result = agent.invoke({"messages": [("user", query)]}, config)
    response = result["messages"][-1].content
    for i in range(max_iter):
        eval_str = (cot_eval | llm | StrOutputParser()).invoke(
            {"question": query, "response": response})
        try:
            ev = json.loads(eval_str)
        except json.JSONDecodeError:
            break
        if ev["score"] >= 8:
            break
        response = (cot_improve | llm | StrOutputParser()).invoke(
            {"question": query, "response": response, "feedback": ev["feedback"]})
    # 개선된 응답을 메모리에 저장
    agent.update_state(config, {"messages": [AIMessage(content=response)]})
    return response

### 실행

In [ ]:
config = {"configurable": {"thread_id": "session-1"}}

# 1차: RAG + Tool Use + Reflexion
result = agent_with_reflexion(
    "내 이름은 민수야. 서울의 연평균 기온을 검색하고 화씨로 변환해줘",
    config)
print(result)

print("\n" + "="*50 + "\n")

# 2차: Memory 활용 (이름 + 계산 결과 기억)
result = agent_with_reflexion(
    "내 이름이 뭐였지? 그리고 아까 계산한 화씨 온도는?",
    config)
print(result)